## Quantize RT-DETR detector to FP16

Downloads the same **1-class** YOLO dataset zip as [finetuning/finetune_rtdetr.ipynb](../finetuning/finetune_rtdetr.ipynb) for validation images, loads `sailcv-rtdetrl640.pt` from Hugging Face, converts weights to half precision, saves a new checkpoint.

**GPU recommended:** On Colab: **Runtime → Change runtime type → GPU**.

**Output:** e.g. `sailcv-rtdetrl640_fp16.pt` in the working directory. Point `detector.model_path` in your parameters YAML to this file (or copy to `checkpoints/`).

In [ ]:
!nvidia-smi

In [ ]:
import os

HOME = os.getcwd()
print(HOME)

## Install dependencies

In [ ]:
!pip install -q "ultralytics>=8.3.0" huggingface_hub opencv-python-headless numpy tqdm
from IPython import display

display.clear_output()
print("OK")

## Download and unzip main dataset from Hugging Face

Same as finetune notebook: `telltale_one_class_fused.zip` from `estefoucher/sail-cv-telltales` (used for optional val-image smoke tests).

In [ ]:
import zipfile
from pathlib import Path

from huggingface_hub import hf_hub_download

main_data_dir = Path(HOME) / "main_data"
main_data_dir.mkdir(parents=True, exist_ok=True)

zip_path = hf_hub_download(
    repo_id="estefoucher/sail-cv-telltales",
    filename="telltale_one_class_fused.zip",
    repo_type="dataset",
    local_dir=HOME,
    local_dir_use_symlinks=False,
)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(main_data_dir)

contents = [p for p in main_data_dir.iterdir() if p.is_dir()]
candidate = main_data_dir
for child in contents:
    if (child / "train").exists() or (child / "val").exists():
        candidate = child
        break
if not (candidate / "train").exists() and not (candidate / "val").exists():
    raise FileNotFoundError(
        f"Expected train/ or val/ under {main_data_dir}. Contents: {list(main_data_dir.iterdir())}"
    )
main_data_dir = candidate.resolve()
print(f"Main dataset at: {main_data_dir}")
for split in ("train", "val"):
    img_dir = main_data_dir / split / "images"
    n = len(list(img_dir.iterdir())) if img_dir.exists() else 0
    print(f"  {split}/images: {n} items")

## Download detector checkpoint from Hugging Face

To use a **local** finetuned `best.pt` instead, set `ckpt_path` to that path and skip the download cell.

In [ ]:
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="estefoucher/tell-tale-detector",
    filename="weights/sailcv-rtdetrl640.pt",
    local_dir=HOME,
    local_dir_use_symlinks=False,
)
print(f"Checkpoint: {ckpt_path}")

## Load, convert to FP16, save

Uses Ultralytics `RTDETR`: move to device, `half()`, then `save()`.

In [ ]:
import torch
from pathlib import Path
from ultralytics import RTDETR

OUT_NAME = "sailcv-rtdetrl640_fp16.pt"
out_path = Path(HOME) / OUT_NAME

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = RTDETR(str(ckpt_path))
model.to(device)
model.half()
model.save(str(out_path))
print(f"Saved FP16 checkpoint: {out_path}")

## Optional: smoke test on val images

Reloads the saved checkpoint and runs inference on a few validation images (640px, same as training defaults).

In [ ]:
import random
import time
from pathlib import Path

import cv2
import torch
from tqdm.auto import tqdm
from ultralytics import RTDETR

val_img_dir = Path(main_data_dir) / "val" / "images"
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif"}
images = [
    p
    for p in val_img_dir.iterdir()
    if p.is_file() and p.suffix.lower() in IMG_EXTS
]
if not images:
    print("No val images found; skip smoke test.")
else:
    sample = random.sample(images, k=min(16, len(images)))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    m = RTDETR(str(out_path))
    m.to(device)
    t0 = time.perf_counter()
    n_det = 0
    for p in tqdm(sample, desc="FP16 infer"):
        im = cv2.imread(str(p))
        if im is None:
            continue
        r = m.predict(im, imgsz=640, verbose=False)
        boxes = r[0].boxes
        if boxes is not None:
            n_det += len(boxes)
    dt = time.perf_counter() - t0
    print(f"Images: {len(sample)}, total boxes: {n_det}, time: {dt:.2f}s")

## Output

Use `sailcv-rtdetrl640_fp16.pt` (or your chosen `OUT_NAME`) as `detector.model_path` in your tracking parameters YAML.